In [3]:
import json
import pandas as pd

In [4]:
with open("./data/hotels_korea_reviews_text.json", encoding="utf-8") as file:
    temp = file.read()
    data = json.loads(temp)

# Hotel info 수집하기

In [5]:
len(data)

9027

In [ ]:
# data[0]

In [ ]:
# 호텔ID
hotel_id = data[0]['data']['body']['pdpHeader']['hotelId']

In [ ]:
# 호텔이름
hotel_name = data[0]['data']['body']['propertyDescription']['name']

In [ ]:
# 호텔 등급
hotel_grade = data[0]['data']['body']['propertyDescription']['starRating']

In [ ]:
# 호텔 지역
hotel_region = data[0]['data']['body']['propertyDescription']['address']['region']

In [ ]:
# 호텔 주소 시도
hotel_sido = data[0]['data']['body']['propertyDescription']['address']['locality']

In [ ]:
# 총리뷰수
n_reviews = data[0]['data']['body']['reviewContent']['overall']['totalCount']

In [ ]:
# 평점
avg_rating = data[0]['data']['body']['reviewContent']['overall']['rating']

In [ ]:
# 세부평점 청결도
cleanliness = data[0]['data']['body']['reviewContent']['overall']['ratingAspects']['cleanliness']
service = data[0]['data']['body']['reviewContent']['overall']['ratingAspects']['service']
comfort = data[0]['data']['body']['reviewContent']['overall']['ratingAspects']['comfort']
condition = data[0]['data']['body']['reviewContent']['overall']['ratingAspects']['condition']
neighbourhood = data[0]['data']['body']['reviewContent']['overall']['ratingAspects']['neighbourhood']


In [ ]:
print(cleanliness, service, comfort, condition, neighbourhood)

In [ ]:
temp ={}
for idx, hotel in enumerate(data):
    print(f"{idx+1}/{len(data)}째 페이지 수집중", end="\r")
    overall = hotel['data']['body']['reviewContent']['overall']
    overall_keys = hotel['data']['body']['reviewContent']['overall'].keys()
    for key in overall_keys:
        if key in ['totalCount','rating']:
            temp.setdefault(key, []).append(overall[key])
        elif key == 'ratingAspects':
            for ratingAspect_key in overall[key].keys():
                temp.setdefault(ratingAspect_key, []).append(overall[key].get(ratingAspect_key, 0.0))
            
temp        

In [ ]:
temp

In [ ]:
%%timeit
hotel_info_result = {}
for idx, hotel in enumerate(data):
    print(f"{idx+1}/{len(data)}째 페이지 수집중", end="\r")
    hotel_id = hotel['data']['body']['pdpHeader']['hotelId']
    hotel_name = hotel['data']['body']['propertyDescription']['name']
    hotel_grade = hotel['data']['body']['propertyDescription']['starRating']
    n_reviews = hotel['data']['body']['reviewContent']['overall']['totalCount']
    avg_rating = hotel['data']['body']['reviewContent']['overall']['rating']
    cleanliness = hotel['data']['body']['reviewContent']['overall']['ratingAspects'].get('cleanliness', 0.0)
    service = hotel['data']['body']['reviewContent']['overall']['ratingAspects'].get('service', 0.0)
    comfort = hotel['data']['body']['reviewContent']['overall']['ratingAspects'].get('comfort', 0.0)
    condition = hotel['data']['body']['reviewContent']['overall']['ratingAspects'].get('condition', 0.0)
    neighbourhood = hotel['data']['body']['reviewContent']['overall']['ratingAspects'].get('neighbourhood', 0.0)
    hotel_sido = hotel['data']['body']['propertyDescription']['address']['locality']
    hotel_region = hotel['data']['body']['propertyDescription']['address'].get('region') if hotel_sido != '서울특별시' else '서울특별시'
    

    hotel_info_result.setdefault("hotel_id", []).append(hotel_id)
    hotel_info_result.setdefault("hotel_name", []).append(hotel_name)
    hotel_info_result.setdefault("hotel_grade", []).append(hotel_grade)
    hotel_info_result.setdefault("n_reviews", []).append(n_reviews)
    hotel_info_result.setdefault("avg_rating", []).append(avg_rating)
    hotel_info_result.setdefault("cleanliness", []).append(cleanliness)
    hotel_info_result.setdefault("service", []).append(service)
    hotel_info_result.setdefault("comfort", []).append(comfort)
    hotel_info_result.setdefault("condition", []).append(condition)
    hotel_info_result.setdefault("neighbourhood", []).append(neighbourhood)
    hotel_info_result.setdefault("hotel_region", []).append(hotel_region)
    hotel_info_result.setdefault("hotel_sido", []).append(hotel_sido)

hotel_info_df = pd.DataFrame(hotel_info_result)
hotel_info_df

* 반복문을 사용해 중복되는 부분 반복문으로 처리하기

In [ ]:
%%timeit
hotel_info_result = {}
for idx, hotel in enumerate(data):
    print(f"{idx+1}/{len(data)}째 페이지 수집중", end="\r")
    hotel_id = hotel['data']['body']['pdpHeader']['hotelId']
    hotel_name = hotel['data']['body']['propertyDescription']['name']
    hotel_grade = hotel['data']['body']['propertyDescription']['starRating']
    # 총리뷰수, 평점, 세부평점  
    overall = hotel['data']['body']['reviewContent']['overall']
    overall_keys = hotel['data']['body']['reviewContent']['overall'].keys()
    for key in overall_keys:
        if key in ['totalCount','rating']:
            hotel_info_result.setdefault(key, []).append(overall[key])
        elif key == 'ratingAspects':
#             for ratingAspect_key in overall[key].keys():
            for ratingAspect_key in ['cleanliness', 'service', 'comfort', 'condition', 'neighbourhood']:
                hotel_info_result.setdefault(ratingAspect_key, []).append(overall[key].get(ratingAspect_key, 0.0))
    hotel_sido = hotel['data']['body']['propertyDescription']['address']['locality']
    hotel_region = hotel['data']['body']['propertyDescription']['address'].get('region') if hotel_sido != '서울특별시' else '서울특별시'
    

    hotel_info_result.setdefault("hotel_id", []).append(hotel_id)
    hotel_info_result.setdefault("hotel_name", []).append(hotel_name)
    hotel_info_result.setdefault("hotel_grade", []).append(hotel_grade)
    hotel_info_result.setdefault("hotel_region", []).append(hotel_region)
    hotel_info_result.setdefault("hotel_sido", []).append(hotel_sido)

hotel_info_df = pd.DataFrame(hotel_info_result)
display(hotel_info_df)

In [ ]:
for key, value in hotel_info_result.items():
    print(f"{key}, {len(value)}")

In [ ]:
data[61]['data']['body']['propertyDescription']['address']

# 사용자 리뷰 수집하기

In [ ]:
hotel_id = data[0]['data']['body']['pdpHeader']['hotelId']
review_list = data[0]['data']['body']["reviewContent"]['reviews']["hermes"]['groups'][0]['items']

['review_id',
 'review_id',
 'review_id',
 'review_id',
 'review_id',
 'review_id',
 'review_id',
 'review_id',
 'review_id',
 'review_id']

In [2]:
reviews = {}
for idx, hotel in enumerate(data[:1]):
    print(f"{idx+1}/{len(data)}째 페이지 수집중", end='\r')
    hotel_id = hotel['data']['body']['pdpHeader']['hotelId']
    key_list = ['itineraryId', 'tripType', 'reviewDate', 'rating', 'description']
    try:    
        review_list = hotel['data']['body']["reviewContent"]['reviews']["hermes"]['groups'][0]['items']
    except Exception as e:
        print(e)
        print("리뷰 데이터가 없어서 다음 호텔로 넘어갑니다.", end="\n\n")
#     reviews['hotel_id'] = hotel_id
    for key in key_list:
        for review in review_list:
            reviews.setdefault('hotel_id', []).append(hotel_id)
            reviews.setdefault(key, []).append(review[key])
review_df = pd.DataFrame(reviews)
review_df

NameError: name 'data' is not defined

In [ ]:
data[2]['data']['body']["reviewContent"]['reviews']["hermes"]

# 호텔정보와 사용자리뷰 동시에 수집하기

In [ ]:
hotel_info_result = {}
all_reviews = []
for idx, hotel in enumerate(data):
    print(f"{idx+1}/{len(data)}째 페이지 수집중", end="\r")
    hotel_id = hotel['data']['body']['pdpHeader']['hotelId']
    hotel_name = hotel['data']['body']['propertyDescription']['name']
    hotel_grade = hotel['data']['body']['propertyDescription']['starRating']
    n_reviews = hotel['data']['body']['reviewContent']['overall']['totalCount']
    avg_rating = hotel['data']['body']['reviewContent']['overall']['rating']
    cleanliness = hotel['data']['body']['reviewContent']['overall']['ratingAspects'].get('cleanliness', 0.0)
    service = hotel['data']['body']['reviewContent']['overall']['ratingAspects'].get('service', 0.0)
    comfort = hotel['data']['body']['reviewContent']['overall']['ratingAspects'].get('comfort', 0.0)
    condition = hotel['data']['body']['reviewContent']['overall']['ratingAspects'].get('condition', 0.0)
    neighbourhood = hotel['data']['body']['reviewContent']['overall']['ratingAspects'].get('neighbourhood', 0.0)
    hotel_sido = hotel['data']['body']['propertyDescription']['address']['locality']
    hotel_region = hotel['data']['body']['propertyDescription']['address'].get('region') if hotel_sido != '서울특별시' else '서울특별시'
    

    hotel_info_result.setdefault("hotel_id", []).append(hotel_id)
    hotel_info_result.setdefault("hotel_name", []).append(hotel_name)
    hotel_info_result.setdefault("hotel_grade", []).append(hotel_grade)
    hotel_info_result.setdefault("n_reviews", []).append(n_reviews)
    hotel_info_result.setdefault("avg_rating", []).append(avg_rating)
    hotel_info_result.setdefault("cleanliness", []).append(cleanliness)
    hotel_info_result.setdefault("service", []).append(service)
    hotel_info_result.setdefault("comfort", []).append(comfort)
    hotel_info_result.setdefault("condition", []).append(condition)
    hotel_info_result.setdefault("neighbourhood", []).append(neighbourhood)
    hotel_info_result.setdefault("hotel_region", []).append(hotel_region)
    hotel_info_result.setdefault("hotel_sido", []).append(hotel_sido)
    
    # 사용자 리뷰 수집 부분
    key_list = ['itineraryId', 'tripType', 'reviewDate', 'rating', 'description']
    try:    
        review_list = hotel['data']['body']["reviewContent"]['reviews']["hermes"]['groups'][0]['items']
    except Exception as e:
        print(e)
        print("리뷰 데이터가 없어서 다음 호텔로 넘어갑니다.", end="\n\n")
    reviews = {}
    reviews['hotel_id'] = hotel_id
    for key in key_list:
        for review in review_list:
            reviews.setdefault(key, []).append(review[key])
    all_reviews.append(pd.DataFrame(reviews))
review_df = pd.concat(all_reviews)
review_df = review_df.reset_index(drop=True)
hotel_info_df = pd.DataFrame(hotel_info_result)


In [ ]:
hotel_info_df.head(3)

In [ ]:
review_df.tail(3)

In [ ]:
review_df[review_df['hotel_id']==1041793024]